In [27]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

#### Recreating the cifar10 dataset images stats mean and std

In [28]:
from pathlib import Path
import numpy as np
from PIL import Image

PATH = Path("../data/cifar10/train/")

# Get all image paths
image_paths = list(PATH.rglob('*.png')) + list(PATH.rglob('*.jpg'))
print(f"Found {len(image_paths)} images")

# Load all images into a single array (vectorized)
images = np.array([np.array(Image.open(p)) for p in image_paths]) / 255.0
# Shape: (num_images, height, width, channels)

# Calculate mean and std across all pixels (vectorized)
MEAN_cf10, std_cf10 = images.mean(axis=(0, 1, 2)), images.std(axis=(0, 1, 2))  # Average & std across images, height, width

print(f"\nMean: {MEAN_cf10}, Std:  {std_cf10}")
print(f"\nstats = (np.array({MEAN_cf10.tolist()}), np.array({std_cf10.tolist()}))")

Found 50000 images

Mean: [0.4914  0.48216 0.44653], Std:  [0.24703 0.24349 0.26159]

stats = (np.array([0.49139967861961836, 0.4821584084000747, 0.4465309144581913]), np.array([0.247032232450512, 0.24348512800087502, 0.26158784173101723]))


In [29]:
from fastai.conv_learner import *
PATH = Path("../data/cifar10/")
os.makedirs(PATH, exist_ok=True)

In [30]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
#stats = (np.array([ 0.4914, 0.48216, 0.44653]), np.array([ 0.24703, 0.24349, 0.26159]))

stats = (np.array([0.49139967861961836, 0.4821584084000747, 0.4465309144581913]), 
         np.array([0.247032232450512, 0.24348512800087502, 0.26158784173101723]))

num_workers = num_cpus()//2
bs=256
sz=32

In [31]:
#after padding it becomes 36x36 image, but then it randomly crops and take 32x32 image
tfms = tfms_from_stats(stats, sz, aug_tfms=[RandomFlip()], pad=sz//8) 
data = ImageClassifierData.from_paths(PATH, val_name='test', tfms=tfms, bs=bs)

flipping and reverse padding instead of black pads, resulting better results and used in fastai
- pad = x[..., -n:].flip(-1)
- x = torch.cat([x, pad], dim=-1)

In [32]:
def conv_layer(n0, nf, ks=3, stride=1):
    return nn.Sequential(
        nn.Conv2d(ni, nf, kernel_size=ks, bias=False, stride=stride, padding=ks//2),
        nn.BatchNorm2d(nf, momentum=0.01),
        nn.LeakyReLU(negative_slope=0.1, inplace=True))

In [34]:
class ResLayer(nn.Module):
    def __init__(self, ni):
        super().__init__()
        self.conv1=conv_layer(ni, ni//2, ks=1)
        self.conv2=conv_layer(ni//2, ni, ks=3)
    def forward(self, x): return self.conv2(self.conv1(x)) + x